# BSP Dungeon Generation

**Domain:** Procedural Generation  ·  *recommended addition*  ·  **runnable:** yes

A compact refresher on **Binary Space Partitioning** for roguelike dungeons: recursively
slice a rectangle into rooms, then stitch them back together with corridors. How it works,
the knobs that matter, where it bites, and what to use instead.

## 1. What & Why

**BSP dungeon generation** builds a dungeon by **recursively cutting a rectangular region in
two** (horizontally or vertically), repeating on each half until the pieces are small enough,
then **carving one room inside each leaf** and **connecting sibling rooms with corridors** on
the way back up the tree. The "BSP" is the same binary space partitioning idea used in
old-school 3D renderers and collision systems — here it just partitions a 2D grid.

**The problem it solves.** You want a dungeon that feels *deliberately laid out* — rooms that
don't overlap, are reasonably spread across the map, and are all reachable — without
hand-placing anything. Naively scattering random rectangles gives you overlaps, clustering,
and orphaned rooms you then have to fix up. BSP gives you **non-overlapping, well-distributed
rooms and guaranteed connectivity for free**, because the tree structure tells you exactly
which rooms are neighbors and the recursion guarantees every region gets used.

**When to reach for it.**

- Classic roguelike / dungeon-crawler levels: discrete rooms joined by hallways.
- You want guaranteed full connectivity without a separate pathfinding/repair pass.
- You want a cheap, deterministic, tunable layout you can seed and reproduce.

**When *not* to.** If you want organic, cave-like spaces, BSP's rectangular bias will fight
you — use **cellular automata** or a **drunkard's walk** instead. If you want loops and
non-tree connectivity (multiple paths between areas), BSP's tree gives you a *minimum*
spanning structure; you must add extra edges yourself.

## 2. Mental Model

Think of **cutting a sheet of paper**. Start with the whole map as one rectangle. Make one
straight cut — horizontal or vertical — splitting it into two child rectangles. Recurse into
each child and cut again. Stop when a rectangle is small enough to be a "room slot". You now
have a **binary tree** whose leaves tile the map with no gaps and no overlaps.

```
          whole map                    binary tree
   +------------------------+              (root)
   |          |             |             /      \
   |    A     |      B      |          split     split
   |          |             |          /  \      /  \
   |----------+-------------|         A    A'   B    B'   <- leaves
   |    A'    |     B'      |         |    |    |    |
   |          |             |        room room room room
   +------------------------+
```

Two phases:

1. **Split (going down).** Recursively partition until leaves are small. Leaves are
   *partitions*, not rooms yet.
2. **Carve + connect (coming back up).** In each leaf, draw a room *smaller than the leaf* so
   rooms have breathing room. Then, at every internal node, **connect the two child subtrees
   with a corridor** (join a room/point from the left subtree to one from the right). Because
   every internal node adds exactly one connection between its two halves, the whole dungeon
   ends up as one connected tree — every room reachable, no separate pass needed.

## 3. Key Concepts

- **Partition tree.** A binary tree of axis-aligned rectangles. Internal nodes are splits;
  leaves are the regions that will hold rooms. The tree *is* the data structure — keep it
  around; you need it for connecting.
- **Split orientation & ratio.** Each cut is horizontal or vertical, usually chosen by aspect
  ratio (split the long axis so rooms don't get too thin) plus randomness. The **split
  position** is picked within a band (e.g. 30%–70%) so partitions aren't all identical.
- **Stop condition (min leaf size).** Recursion stops when a region is below a minimum size
  (or a max depth is hit). This **controls room count and size**: smaller minimum → more,
  smaller rooms.
- **Room inset / padding.** Rooms are carved *inside* their leaf with a margin, so adjacent
  rooms don't touch and corridors are visible. Room size is randomized within the leaf.
- **Corridor connection.** Walking back up, each node connects its two children. Common
  choices: connect the **center points** of the two subtrees' rooms, using an **L-shaped
  (dogleg) corridor** (one horizontal + one vertical segment) or a straight one when aligned.
- **Connectivity guarantee.** With one connection per internal node and `n` leaves, you get a
  spanning tree over the rooms — **fully connected, exactly one path between any two rooms**
  (a tree, no loops) unless you add extra corridors deliberately.

## 4. Setup

Pure Python + NumPy for the grid; Matplotlib only to visualize. All CPU-only and tiny — a
60×60 dungeon runs instantly. No external assets or downloads.

In [ ]:
# %pip install numpy matplotlib
import numpy as np

rng = np.random.default_rng(7)  # fixed seed -> reproducible dungeon
print("numpy", np.__version__)

## 5. Worked Examples

### Example 1 — partition, carve, connect (end to end)

A direct, readable implementation. A `Leaf` is a rectangle that can split into two children.
We recursively split, carve a randomized room into each leaf, then connect rooms back up the
tree with L-shaped corridors. The grid uses `1` for floor and `0` for wall.

In [ ]:
WALL, FLOOR = 0, 1

class Leaf:
    def __init__(self, x, y, w, h):
        self.x, self.y, self.w, self.h = x, y, w, h
        self.left = self.right = None
        self.room = None  # (rx, ry, rw, rh) once carved

    def split(self, min_size, rng):
        if self.left or self.right:
            return False
        # Decide orientation: prefer splitting the longer axis.
        if self.w / self.h >= 1.25:
            horizontal = False           # cut vertically (split width)
        elif self.h / self.w >= 1.25:
            horizontal = True            # cut horizontally (split height)
        else:
            horizontal = rng.random() < 0.5

        max_extent = (self.h if horizontal else self.w) - min_size
        if max_extent <= min_size:
            return False                 # too small to split usefully
        cut = rng.integers(min_size, max_extent + 1)

        if horizontal:
            self.left  = Leaf(self.x, self.y, self.w, cut)
            self.right = Leaf(self.x, self.y + cut, self.w, self.h - cut)
        else:
            self.left  = Leaf(self.x, self.y, cut, self.h)
            self.right = Leaf(self.x + cut, self.y, self.w - cut, self.h)
        return True


def build_tree(leaf, min_size, max_depth, rng, depth=0):
    if depth < max_depth and leaf.split(min_size, rng):
        build_tree(leaf.left,  min_size, max_depth, rng, depth + 1)
        build_tree(leaf.right, min_size, max_depth, rng, depth + 1)


def carve_rooms(leaf, grid, rng, inset=1):
    """Carve a randomized room into every leaf; return room rects bottom-up."""
    if leaf.left or leaf.right:
        for child in (leaf.left, leaf.right):
            if child:
                carve_rooms(child, grid, rng, inset)
        return
    # Leaf: pick a room smaller than the partition, with a margin.
    rw = int(rng.integers(max(3, leaf.w - 4), leaf.w - inset))
    rh = int(rng.integers(max(3, leaf.h - 4), leaf.h - inset))
    rw, rh = min(rw, leaf.w - inset - 1), min(rh, leaf.h - inset - 1)
    rx = leaf.x + int(rng.integers(inset, max(inset + 1, leaf.w - rw)))
    ry = leaf.y + int(rng.integers(inset, max(inset + 1, leaf.h - rh)))
    leaf.room = (rx, ry, rw, rh)
    grid[ry:ry + rh, rx:rx + rw] = FLOOR


def center(leaf):
    """Center point of this subtree's representative room (carve-side aware)."""
    if leaf.room is not None:
        rx, ry, rw, rh = leaf.room
        return (rx + rw // 2, ry + rh // 2)
    # internal node: borrow a child's room center
    for child in (leaf.left, leaf.right):
        if child:
            return center(child)
    return (leaf.x + leaf.w // 2, leaf.y + leaf.h // 2)


def h_corridor(grid, x1, x2, y):
    grid[y, min(x1, x2):max(x1, x2) + 1] = FLOOR

def v_corridor(grid, y1, y2, x):
    grid[min(y1, y2):max(y1, y2) + 1, x] = FLOOR


def connect(leaf, grid, rng):
    """Walk up the tree; join each node's two children with an L-shaped corridor."""
    if not (leaf.left or leaf.right):
        return
    for child in (leaf.left, leaf.right):
        if child:
            connect(child, grid, rng)
    if leaf.left and leaf.right:
        x1, y1 = center(leaf.left)
        x2, y2 = center(leaf.right)
        if rng.random() < 0.5:           # horizontal-then-vertical, or vice versa
            h_corridor(grid, x1, x2, y1)
            v_corridor(grid, y1, y2, x2)
        else:
            v_corridor(grid, y1, y2, x1)
            h_corridor(grid, x1, x2, y2)


def generate_dungeon(w, h, min_size=10, max_depth=5, rng=rng):
    grid = np.zeros((h, w), dtype=int)
    root = Leaf(0, 0, w, h)
    build_tree(root, min_size, max_depth, rng)
    carve_rooms(root, grid, rng)
    connect(root, grid, rng)
    return grid, root


def count_leaves(leaf):
    if not (leaf.left or leaf.right):
        return 1
    return sum(count_leaves(c) for c in (leaf.left, leaf.right) if c)


grid, root = generate_dungeon(60, 40, min_size=10, max_depth=5,
                              rng=np.random.default_rng(7))
print("grid shape :", grid.shape)
print("rooms      :", count_leaves(root))
print("floor cells:", int(grid.sum()), f"({grid.mean():.1%} of map)")

A quick ASCII peek confirms discrete rooms joined by thin corridors (`#` = wall, `.` = floor):

In [ ]:
def ascii_render(grid):
    return "\n".join("".join("." if c else "#" for c in row) for row in grid)

print(ascii_render(grid[:20, :60]))  # top slice, keeps it readable

Now render the full dungeon. Because we kept the partition tree, we can also overlay the BSP
cuts to *see* how the rooms map onto leaves.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; no display server needed
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def leaf_rects(leaf, out):
    if leaf.left or leaf.right:
        for c in (leaf.left, leaf.right):
            if c:
                leaf_rects(c, out)
    else:
        out.append((leaf.x, leaf.y, leaf.w, leaf.h))
    return out

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(grid, cmap="binary", origin="upper", interpolation="nearest")
for (x, y, w, h) in leaf_rects(root, []):
    ax.add_patch(Rectangle((x - 0.5, y - 0.5), w, h, fill=False,
                           edgecolor="tab:red", lw=0.8, alpha=0.6))
ax.set_title("BSP dungeon — floors (black) with leaf partitions (red)")
ax.axis("off")
fig.tight_layout()
fig.savefig("bsp_dungeon.png", dpi=90)
print("saved bsp_dungeon.png")

### Example 2 — `min_size` is the knob that sets room count

The minimum leaf size controls how deep the recursion goes, and therefore how many rooms you
get and how big they are. Smaller minimum → deeper tree → more, smaller rooms. We sweep it and
measure room count and floor coverage on a fixed seed.

In [ ]:
def dungeon_stats(min_size, seed=7, w=60, h=40, max_depth=8):
    g, r = generate_dungeon(w, h, min_size=min_size, max_depth=max_depth,
                            rng=np.random.default_rng(seed))
    return count_leaves(r), g.mean()

print(f"{'min_size':>9} | {'rooms':>5} | {'floor coverage':>14}")
print("-" * 34)
for ms in (8, 12, 16, 22, 30):
    rooms, cov = dungeon_stats(ms)
    print(f"{ms:>9} | {rooms:>5} | {cov:>13.1%}")

print("\nSmaller min_size -> deeper splits -> more (smaller) rooms.")

As `min_size` shrinks, the partition tree gets deeper, producing many small rooms; as it
grows, you get a handful of large rooms. Floor coverage *drops* as rooms multiply — each leaf
keeps its inset margin, so more partitions means more wall between rooms (and at the extreme,
one giant room nearly fills the map). This single knob (plus `max_depth` as a hard ceiling)
is usually all you tune for layout granularity.

## 6. Gotchas & Pitfalls

- **Rooms filling the whole leaf.** If you carve a room exactly the size of its partition,
  adjacent rooms touch and the dungeon reads as one blob. Always **inset** the room inside the
  leaf (random size + random offset within a margin) so corridors are visible and rooms feel
  distinct.
- **Splitting into slivers.** Cutting too close to an edge yields a 1–2 wide partition that
  can't hold a room. Guard the cut position with a minimum band (here `rng.integers(min_size,
  max_extent+1)`) and bias the cut to the **long axis** so partitions stay roughly square.
- **Corridors that don't reach the room.** Connecting *leaf centers* instead of *room centers*
  can route a hallway through wall space and miss the room. Connect the actual carved room
  centers (or clamp the corridor endpoint into the room).
- **Lost connectivity from connecting the wrong nodes.** The guarantee only holds if you
  connect the **two children of each internal node**. Connecting random leaf pairs, or
  connecting only leaves, can leave subtrees isolated. Do the connection **bottom-up on the
  tree**.
- **It's a tree, so there are no loops.** Exactly one path between any two rooms. That can feel
  linear/mazey. If you want tactical loops, add a few **extra corridors** between nearby rooms
  after the tree pass (an explicit, deliberate step).
- **Everything is rectangular and axis-aligned.** BSP has a strong blocky bias. If you need
  organic shapes, post-process (cellular-automata smoothing) or pick a different algorithm.
- **Off-by-one / orientation bugs.** NumPy grids are `[row, col]` = `[y, x]`. Mixing up axes
  when carving rooms or drawing corridors is the most common source of holes in walls — keep a
  single convention (`grid[y, x]`) everywhere.

## 7. When to Use vs Alternatives

| Approach | Strengths | Weaknesses | Reach for it when |
|---|---|---|---|
| **BSP partitioning** | Non-overlapping rooms, even spread, connectivity for free, cheap & tunable | Rectangular/blocky, tree = no loops, can feel gridded | Classic room-and-corridor dungeons with guaranteed reachability |
| **Cellular automata** | Organic cave shapes, natural blobs | No discrete rooms; needs a connectivity/flood-fill repair pass | Caves, caverns, organic interiors |
| **Drunkard's walk** | Dead simple, winding organic tunnels | Uneven coverage, no rooms, can wander off | Mines, twisty corridors, quick prototypes |
| **Random room placement + graph** | Full control over room shapes/loops | Must handle overlaps and connectivity yourself | You need irregular rooms and explicit loop control |
| **Delaunay + MST (+ extra edges)** | Natural room graph with tunable loopiness | More machinery; rooms placed separately | You want organic connectivity with some loops |

**Rule of thumb:** reach for **BSP** when you want the tidy, deliberate *room-and-hallway*
feel with zero connectivity headaches. Switch to **cellular automata** or a **drunkard's
walk** when you want organic caves, and to a **Delaunay/MST graph** when you want controllable
loops between separately-shaped rooms. See the Drunkard's Walk and Cellular Automata notebooks
in this domain for the organic alternatives.

## 8. Resources

- **RogueBasin — Basic BSP Dungeon generation** (the canonical tutorial with pseudocode and
  diagrams): <http://www.roguebasin.com/index.php/Basic_BSP_Dungeon_generation>
- **Red Blob Games — guidance on procedural map generation / dungeon connectivity**:
  <https://www.redblobgames.com/maps/>
- **Bob Nystrom — "Rooms and Mazes" dungeon generator** (a different, complementary take on
  rooms + connectivity worth contrasting with BSP):
  <https://journal.stuffwithstuff.com/2014/12/21/rooms-and-mazes/>
- **Wikipedia — Binary space partitioning** (the underlying data structure):
  <https://en.wikipedia.org/wiki/Binary_space_partitioning>
- **TCOD / python-tcod roguelike tutorial — "Generating a dungeon"** (a complete, modern
  Python implementation in context): <https://rogueliketutorials.com/tutorials/tcod/v2/part-3/>